In [1]:
from langchain_anthropic import ChatAnthropic
from langchain_community.tools import DuckDuckGoSearchRun, WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.agents import create_tool_calling_agent, AgentExecutor
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from dotenv import load_dotenv

load_dotenv()

llm = ChatAnthropic(model="claude-sonnet-4-6")

# Tools
tools = [
    DuckDuckGoSearchRun(),
    WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
]

# Prompt with memory placeholder
prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an intelligent research assistant.
    You have access to web search and Wikipedia.
    Always think step by step before answering.
    Be concise and cite your sources."""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}")
])

# Build agent
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Conversation with memory
chat_history = []

def run_agent(user_input):
    result = agent_executor.invoke({
        "input": user_input,
        "chat_history": chat_history
    })
    chat_history.append(HumanMessage(content=user_input))
    chat_history.append(AIMessage(content=result["output"]))
    return result["output"]

# Test the agent
print(run_agent("What is LangChain and who created it?"))
print(run_agent("How does it compare to AutoGen?"))

ImportError: cannot import name 'create_tool_calling_agent' from 'langchain.agents' (/Software/langenv/lib/python3.12/site-packages/langchain/agents/__init__.py)